# Web scrapping game reviews from a webpage

Web scraping is collecting or copy data from websites automatically. This tutorial is a short example about collecting reviews for a game from a webpage. The content the code here is just adapted from the tutorial here: https://towardsdatascience.com/web-scraping-metacritic-reviews-using-beautifulsoup-63801bbe200e. 

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

First we need to Request the html page through the url. Don't forget to assign headers when you request the URL, or your request may be recjectd in some cases.

In [2]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/70.0.3538.77 Safari/537.36'}

The status code 200 means your request is accept.

In [3]:
game = requests.get('https://www.metacritic.com/game/switch/pokemon-sword/user-reviews?page=0',headers = headers)
game.status_code

200

Then we create the beautiful soup objective "soup" using the parser "html.parser". The "soup" is a parsed html document. 

In [4]:
soup = BeautifulSoup(game.text, 'html.parser')
# soup

Then we create a dictionary to hold reviews and the rating.

In [5]:
review_dict = {'review':[],'rating':[]}

Always inpect the html code to find the html structure of the content you are interested before you do the scrapping!

Let's get the "review_section" first reviewer. 

In [6]:
# 'soup.find_all' will navigate the beautiful soup objective and get all the 'review_content' blocks
first_review = soup.find_all('div', class_='review_content')[0]
# print(first_review)
# review = soup.find_all(lambda tag: tag.name == 'div' and tag.get('class') == ['review_section'])[1]


Then get the rating score from this review_section.

In [7]:
score_block = first_review.find('div', class_='review_grade')
# print(score_block)
# score_block.find('div')
score = int(score_block.find('div').text)
print(score)

5


Then get the whole review content from this review_section.

In [8]:
if first_review.find('span', class_='blurb blurb_expanded'):
    review_content = first_review.find('span', class_='blurb blurb_expanded').text
else:
    review_content = first_review.find('div', class_='review_body').find('span').text
# review_content

Lets loop over first 10 pages to get all the reviews and corresponding rating scores. 

In [9]:
for page in range(0,10): 
    url = 'https://www.metacritic.com/game/switch/pokemon-sword/user-reviews?page={}'.format(page)
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/70.0.3538.77 Safari/537.36'}
    game  = requests.get(url, headers = headers)
    soup = BeautifulSoup(game.text, 'html.parser')
    for review in soup.find_all('div', class_='review_content'):
        # if review.find('div', class_='name') == None:
        #     break 
        review_dict['rating'].append(review.find('div', class_='review_grade').find('div').text)
        if review.find('span', class_='blurb blurb_expanded'):
            review_dict['review'].append(review.find('span', class_='blurb blurb_expanded').text)
        elif review.find('div', class_='review_body').find('span'):
             review_dict['review'].append(review.find('div', class_='review_body').find('span').text)
        else:
             review_dict['review'].append(np.nan)

In [10]:
reviews_df = pd.DataFrame(review_dict)  
display(reviews_df)

,review,rating
0,My copy of Sword and Shield came in the mail a...,5
1,The game is meh not even counting the Pokemon ...,5
2,This is what happen if a company realize that ...,3
3,I really wish the games were good. I don't lea...,3
4,"The critics calling Sword and Sheild ""the best...",0
...,...,...
1025,Lazy cashgrab garbage with no redeeming factor...,0
1026,They cut about 400 pokemon for absolutely no r...,0
1027,NaN,80
1028,NaN,75


In [11]:
reviews_df.loc[reviews_df["review"].isnull()]

,review,rating
100,NaN,80
101,NaN,75
102,NaN,78
203,NaN,80
204,NaN,75
205,NaN,78
306,NaN,80
307,NaN,75
308,NaN,78
409,NaN,80
